In [1]:
# Copyright 2021 DeepMind Technologies Limited
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

In [2]:
# Install dependencies for Google Colab.
# If you want to run this notebook on your own machine, you can skip this cell
!pip install dm-haiku
!pip install einops

!mkdir /content/perceiver
!touch /content/perceiver/__init__.py
!wget -O /content/perceiver/bytes_tokenizer.py https://raw.githubusercontent.com/deepmind/deepmind-research/master/perceiver/bytes_tokenizer.py
!wget -O /content/perceiver/io_processors.py https://raw.githubusercontent.com/deepmind/deepmind-research/master/perceiver/io_processors.py
!wget -O /content/perceiver/perceiver.py https://raw.githubusercontent.com/deepmind/deepmind-research/master/perceiver/perceiver.py
!wget -O /content/perceiver/position_encoding.py https://raw.githubusercontent.com/deepmind/deepmind-research/master/perceiver/position_encoding.py

     |████████████████████████████████| 287 kB 5.4 MB/s 
--2022-01-08 15:40:07--  https://raw.githubusercontent.com/deepmind/deepmind-research/master/perceiver/bytes_tokenizer.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1850 (1.8K) [text/plain]
Saving to: ‘/content/perceiver/bytes_tokenizer.py’

/content/perceiver/ 100%[===================>]   1.81K  --.-KB/s    in 0s      

2022-01-08 15:40:07 (19.9 MB/s) - ‘/content/perceiver/bytes_tokenizer.py’ saved [1850/1850]

--2022-01-08 15:40:07--  https://raw.githubusercontent.com/deepmind/deepmind-research/master/perceiver/io_processors.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.git

In [3]:
#@title Import
from typing import Union

import haiku as hk
import jax
import jax.numpy as jnp
import numpy as np
import pickle

from perceiver import perceiver, position_encoding, io_processors, bytes_tokenizer

In [4]:
#@title Load parameters from checkpoint
!wget -O language_perceiver_io_bytes.pickle https://storage.googleapis.com/perceiver_io/language_perceiver_io_bytes.pickle

with open("language_perceiver_io_bytes.pickle", "rb") as f:
  params = pickle.loads(f.read())

--2022-01-08 15:40:15--  https://storage.googleapis.com/perceiver_io/language_perceiver_io_bytes.pickle
Resolving storage.googleapis.com (storage.googleapis.com)... 108.177.120.128, 142.250.128.128, 142.251.6.128, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|108.177.120.128|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 804479532 (767M) [application/octet-stream]
Saving to: ‘language_perceiver_io_bytes.pickle’

language_perceiver_ 100%[===================>] 767.21M   160MB/s    in 5.3s    

2022-01-08 15:40:20 (145 MB/s) - ‘language_perceiver_io_bytes.pickle’ saved [804479532/804479532]



In [5]:
#@title Model config
D_MODEL = 768
D_LATENTS = 1280
MAX_SEQ_LEN = 2048

encoder_config = dict(
    num_self_attends_per_block=26,
    num_blocks=1,
    z_index_dim=256,
    num_z_channels=D_LATENTS,
    num_self_attend_heads=8,
    num_cross_attend_heads=8,
    qk_channels=8 * 32,
    v_channels=D_LATENTS,
    use_query_residual=True,
    cross_attend_widening_factor=1,
    self_attend_widening_factor=1)

decoder_config = dict(
    output_num_channels=D_LATENTS,
    position_encoding_type='trainable',
    output_index_dims=MAX_SEQ_LEN,
    num_z_channels=D_LATENTS,
    qk_channels=8 * 32,
    v_channels=D_MODEL,
    num_heads=8,
    final_project=False,
    use_query_residual=False,
    trainable_position_encoding_kwargs=dict(num_channels=D_MODEL))

# The tokenizer is just UTF-8 encoding (with an offset)
tokenizer = bytes_tokenizer.BytesTokenizer()

In [6]:
#@title Decoding Perceiver Model
def apply_perceiver(
    inputs: jnp.ndarray, input_mask: jnp.ndarray) -> jnp.ndarray:
  """Runs a forward pass on the Perceiver.

  Args:
    inputs: input bytes, an int array of shape [B, T]
    input_mask: Array of shape indicating which entries are valid and which are
      masked. A truthy value indicates that the entry is valid.

  Returns:
    The output logits, an array of shape [B, T, vocab_size].
  """
  assert inputs.shape[1] == MAX_SEQ_LEN

  embedding_layer = hk.Embed(
      vocab_size=tokenizer.vocab_size,
      embed_dim=D_MODEL)
  embedded_inputs = embedding_layer(inputs)

  batch_size = embedded_inputs.shape[0]

  input_pos_encoding = perceiver.position_encoding.TrainablePositionEncoding(
      index_dim=MAX_SEQ_LEN, num_channels=D_MODEL)
  embedded_inputs = embedded_inputs + input_pos_encoding(batch_size)
  perceiver_mod = perceiver.Perceiver(
      encoder=perceiver.PerceiverEncoder(**encoder_config),
      decoder=perceiver.BasicDecoder(**decoder_config))
  output_embeddings = perceiver_mod(
      embedded_inputs, is_training=False, input_mask=input_mask, query_mask=input_mask)

  logits = io_processors.EmbeddingDecoder(
      embedding_matrix=embedding_layer.embeddings)(output_embeddings)
  return logits

apply_perceiver = hk.transform(apply_perceiver).apply

In [7]:
input_str = "This is an incomplete sentence where some words are missing."
input_tokens = tokenizer.to_int(input_str)

# Mask " missing.". Note that the model performs much better if the masked chunk
# starts with a space.
input_tokens[51:60] = tokenizer.mask_token
print("Tokenized string without masked bytes:")
print(tokenizer.to_string(input_tokens))

Tokenized string without masked bytes:
This is an incomplete sentence where some words are


In [8]:
#@title Pad and reshape inputs
inputs = input_tokens[None]
input_mask = np.ones_like(inputs)

def pad(max_sequence_length: int, inputs, input_mask):
  input_len = inputs.shape[1]
  assert input_len <= max_sequence_length
  pad_len = max_sequence_length - input_len
  padded_inputs = np.pad(
      inputs,
      pad_width=((0, 0), (0, pad_len)),
      constant_values=tokenizer.pad_token)
  padded_mask = np.pad(
      input_mask,
      pad_width=((0, 0), (0, pad_len)),
      constant_values=0)
  return padded_inputs, padded_mask

inputs, input_mask = pad(MAX_SEQ_LEN, inputs, input_mask)

In [9]:
rng = jax.random.PRNGKey(1)  # Unused

out = apply_perceiver(params, rng=rng, inputs=inputs, input_mask=input_mask)

masked_tokens_predictions = out[0, 51:60].argmax(axis=-1)
print("Greedy predictions:")
print(masked_tokens_predictions)
print()
print("Predicted string:")
print(tokenizer.to_string(masked_tokens_predictions))

Greedy predictions:
[ 38 115 111 121 121 111 116 109  52]

Predicted string:
 missing.


### Try
* The first oranges weren't orange. (failed)
  * bcz single quote `'`
* The first oranges were not orange. (failed)
  * prediction : "picked."
* A game played between two or four people on a special playing area that involves hitting a small ball across a central net using a racket. (failed)
  * prediction : " bat e ."
* This latest injury must surely mean that her tennis career is now at an end. (passed)
  * prediction : " end."  


In [10]:
input_str2 = "The first oranges were not orange."
input_tokens2 = tokenizer.to_int(input_str2)

last_sep = list(filter(lambda irow: irow[1]==' ', enumerate(input_str2)))[-1][0]
print(input_str2[last_sep:])
len_str2 = len(input_str2)

input_tokens2[last_sep:len_str2] = tokenizer.mask_token
print("Tokenized string without masked bytes:")
print(tokenizer.to_string(input_tokens2))

 orange.
Tokenized string without masked bytes:
The first oranges were not


In [11]:
inputs2 = input_tokens2[None]
input_mask2 = np.ones_like(inputs2)
inputs2, input_mask2 = pad(MAX_SEQ_LEN, inputs2, input_mask2)

out2 = apply_perceiver(params, rng=rng, inputs=inputs2, input_mask=input_mask2)

masked_tokens_predictions2 = out2[0, last_sep:len_str2].argmax(axis=-1)
print("Greedy predictions:")
print(masked_tokens_predictions2)
print()
print("Predicted string:")
print(tokenizer.to_string(masked_tokens_predictions2))

Greedy predictions:
[ 38 118 111 105 113 107 106  52]

Predicted string:
 picked.


In [12]:
input_str3 = "A game played between two or four people on a special playing area that involves hitting a small ball across a central net using a racket."
input_tokens3 = tokenizer.to_int(input_str3)

last_sep = list(filter(lambda irow: irow[1]==' ', enumerate(input_str3)))[-1][0]
print(input_str3[last_sep:])
len_str3 = len(input_str3)

input_tokens3[last_sep:len_str3] = tokenizer.mask_token
print("Tokenized string without masked bytes:")
print(tokenizer.to_string(input_tokens3))

 racket.
Tokenized string without masked bytes:
A game played between two or four people on a special playing area that involves hitting a small ball across a central net using a


In [13]:
inputs3 = input_tokens3[None]
input_mask3 = np.ones_like(inputs3)
inputs3, input_mask3 = pad(MAX_SEQ_LEN, inputs3, input_mask3)

out3 = apply_perceiver(params, rng=rng, inputs=inputs3, input_mask=input_mask3)

masked_tokens_predictions3 = out3[0, last_sep:len_str3].argmax(axis=-1)
print("Greedy predictions:")
print(masked_tokens_predictions3)
print()
print("Predicted string:")
print(tokenizer.to_string(masked_tokens_predictions3))

Greedy predictions:
[ 38 104 103 122  38 107  38  52]

Predicted string:
 bat e .


In [14]:
input_str4 = "This latest injury must surely mean that her tennis career is now at an end."
input_tokens4 = tokenizer.to_int(input_str4)

last_sep = list(filter(lambda irow: irow[1]==' ', enumerate(input_str4)))[-1][0]
print(input_str4[last_sep:])
len_str4 = len(input_str4)

input_tokens4[last_sep:len_str4] = tokenizer.mask_token
print("Tokenized string without masked bytes:")
print(tokenizer.to_string(input_tokens4))

 end.
Tokenized string without masked bytes:
This latest injury must surely mean that her tennis career is now at an


In [15]:
inputs4 = input_tokens4[None]
input_mask4 = np.ones_like(inputs4)
inputs4, input_mask4 = pad(MAX_SEQ_LEN, inputs4, input_mask4)

out4 = apply_perceiver(params, rng=rng, inputs=inputs4, input_mask=input_mask4)

masked_tokens_predictions4 = out4[0, last_sep:len_str4].argmax(axis=-1)
print("Greedy predictions:")
print(masked_tokens_predictions4)
print()
print("Predicted string:")
print(tokenizer.to_string(masked_tokens_predictions4))

Greedy predictions:
[ 38 107 116 106  52]

Predicted string:
 end.


### Checks

#### Check Model

In [ ]:
print('\n'.join(params.keys()))

trainable_position_encoding
basic_decoder/~/trainable_position_encoding
perceiver_encoder/~/trainable_position_encoding
perceiver_encoder/~/cross_attention/attention/linear_1
perceiver_encoder/~/cross_attention/attention/linear_3
perceiver_encoder/~/cross_attention/attention/linear
perceiver_encoder/~/cross_attention/attention/linear_2
perceiver_encoder/~/cross_attention/layer_norm
perceiver_encoder/~/cross_attention/layer_norm_1
perceiver_encoder/~/cross_attention/layer_norm_2
perceiver_encoder/~/cross_attention/mlp/linear
perceiver_encoder/~/cross_attention/mlp/linear_1
basic_decoder/cross_attention/attention/linear_1
basic_decoder/cross_attention/attention/linear_3
basic_decoder/cross_attention/attention/linear
basic_decoder/cross_attention/attention/linear_2
basic_decoder/cross_attention/layer_norm
basic_decoder/cross_attention/layer_norm_1
basic_decoder/cross_attention/layer_norm_2
basic_decoder/cross_attention/mlp/linear
basic_decoder/cross_attention/mlp/linear_1
perceiver_encode

In [ ]:
print(params['embed']['embeddings'].shape)
params['embed']['embeddings']

(262, 768)


array([[-0.0522982 , -0.0105989 , -0.05496693, ...,  0.09212864,
         0.01774625,  0.00090622],
       [-0.07013632, -0.07802165, -0.00419141, ...,  0.04256282,
         0.01444387, -0.04895917],
       [-0.02526611, -0.07566191, -0.06929566, ...,  0.08521955,
        -0.03334047, -0.04332254],
       ...,
       [-0.10163219, -0.12031052, -0.15186352, ...,  0.01399962,
         0.00024765, -0.07067219],
       [-0.03462069, -0.06277017, -0.13482094, ..., -0.00067478,
         0.04591347, -0.07697366],
       [-0.0194727 , -0.00323541, -0.06477115, ...,  0.08492611,
        -0.05093516, -0.00701316]], dtype=float32)

In [ ]:
print(params['embedding_decoder']['bias'].shape)
params['embedding_decoder']['bias']

(262,)


array([-12.817861  , -12.766624  , -12.743388  , -14.813668  ,
       -12.757696  , -12.766587  , -10.750544  , -10.778125  ,
       -10.598178  ,  -8.783736  , -11.388426  , -11.256775  ,
       -11.230276  , -10.09811   , -10.479769  ,  -4.281828  ,
        -0.31707364, -12.838216  , -12.840149  , -12.836245  ,
       -11.29228   , -10.956255  , -10.741253  , -11.163673  ,
       -11.172848  , -10.877851  , -10.787847  , -11.069781  ,
       -11.329424  , -11.248469  , -10.582684  , -10.795291  ,
       -10.4019575 , -10.166274  , -12.916149  , -12.909197  ,
       -12.813203  , -10.202336  ,   1.4839268 ,  -2.4843597 ,
        -1.1889938 ,  -3.1233952 ,  -3.3400922 ,  -3.5407178 ,
        -3.429324  ,  -1.2496376 ,  -1.160968  ,  -0.8967215 ,
        -2.528663  ,  -3.1994536 ,  -0.5383367 ,   0.3578812 ,
        -0.31536317,  -0.85231984,  -1.2259562 ,  -1.0152757 ,
        -1.2200558 ,  -1.6727653 ,  -1.8017595 ,  -1.7060462 ,
        -2.1528652 ,  -1.9717485 ,  -1.9443219 ,  -2.06

#### Check `tokenizer`

In [ ]:
tokenizer.vocab_size

262

In [ ]:
tokenizer.mask_token

3

In [ ]:
tokenizer.pad_token

0

In [ ]:
tokenizer.mask_token

3

In [ ]:
tokenizer.to_int("test one two")

array([122, 107, 121, 122,  38, 117, 116, 107,  38, 122, 125, 117],
      dtype=int32)

In [ ]:
tokenizer.to_int("test one two .")

array([122, 107, 121, 122,  38, 117, 116, 107,  38, 122, 125, 117,  38,
        52], dtype=int32)

In [ ]:
tokenizer.to_string(np.array([122, 107, 121, 122,  38, 117]))

'test o'

In [ ]:
tokenizer.to_string(np.array([122, 107, 121, 122,  38, 117, 116]))

'test on'

#### Check pad

In [ ]:
print(input_tokens.shape)
input_tokens

(60,)


array([ 90, 110, 111, 121,  38, 111, 121,  38, 103, 116,  38, 111, 116,
       105, 117, 115, 118, 114, 107, 122, 107,  38, 121, 107, 116, 122,
       107, 116, 105, 107,  38, 125, 110, 107, 120, 107,  38, 121, 117,
       115, 107,  38, 125, 117, 120, 106, 121,  38, 103, 120, 107,   3,
         3,   3,   3,   3,   3,   3,   3,   3], dtype=int32)

In [ ]:
print(MAX_SEQ_LEN)
print(input_tokens[None].shape[1])
print('pad_len = ', MAX_SEQ_LEN - input_tokens[None].shape[1])

2048
60
pad_len =  1988


In [ ]:
np.pad(
    input_tokens[None],
    pad_width=((0, 0), (0, MAX_SEQ_LEN - input_tokens[None].shape[1])),
    constant_values=tokenizer.pad_token)

array([[ 90, 110, 111, ...,   0,   0,   0]], dtype=int32)

In [ ]:
np.pad(
    np.ones_like(input_tokens[None]),
    pad_width=((0, 0), (0, MAX_SEQ_LEN - input_tokens[None].shape[1])),
    constant_values=0)

array([[1, 1, 1, ..., 0, 0, 0]], dtype=int32)

#### Check `embedded_inputs`

In [ ]:
D_MODEL

768

In [ ]:
tmp_pad_inputs = np.pad(
    input_tokens[None],
    pad_width=((0, 0), (0, MAX_SEQ_LEN - input_tokens[None].shape[1])),
    constant_values=tokenizer.pad_token)

In [ ]:
def tmp_layer_operator(inputs: jnp.ndarray) -> jnp.ndarray:

  assert inputs.shape[1] == MAX_SEQ_LEN
  tmp_layer = hk.Embed(
    vocab_size=tokenizer.vocab_size,
    embed_dim=D_MODEL)
  tmp_embedded_inputs = tmp_layer(inputs)
  return tmp_embedded_inputs

tmp_layer_operator = hk.transform(tmp_layer_operator).apply

In [ ]:
 tmp_operated_embedded_inputs = tmp_layer_operator(params, rng=rng, inputs=tmp_pad_inputs)

In [ ]:
print(tmp_operated_embedded_inputs.shape)
print('batch_size :', tmp_operated_embedded_inputs.shape[0])
tmp_operated_embedded_inputs

(1, 2048, 768)
batch_size : 1


DeviceArray([[[ 0.06206276,  0.06097331, -0.01906533, ...,  0.03507749,
               -0.06031232,  0.0292527 ],
              [ 0.02303478,  0.030209  , -0.01698836, ...,  0.00549698,
               -0.0274268 ,  0.04424436],
              [ 0.01308858,  0.00334069, -0.02062531, ...,  0.02043347,
               -0.0160682 ,  0.05579976],
              ...,
              [-0.0522982 , -0.0105989 , -0.05496693, ...,  0.09212864,
                0.01774625,  0.00090622],
              [-0.0522982 , -0.0105989 , -0.05496693, ...,  0.09212864,
                0.01774625,  0.00090622],
              [-0.0522982 , -0.0105989 , -0.05496693, ...,  0.09212864,
                0.01774625,  0.00090622]]], dtype=float32)

#### Check Trainable Position Encoding

In [ ]:
def tmp_pos_encoding_operator(batch_size: int) -> jnp.ndarray:

  input_pos_encoding = perceiver.position_encoding.TrainablePositionEncoding(
      index_dim=MAX_SEQ_LEN, num_channels=D_MODEL)
  return input_pos_encoding(batch_size)

tmp_pos_encoding_operator = hk.transform(tmp_pos_encoding_operator).apply  

In [ ]:
tmp_encoded_pos = tmp_pos_encoding_operator(params, batch_size=1, rng=rng)

In [ ]:
print(tmp_encoded_pos.shape)
tmp_encoded_pos

(1, 2048, 768)


array([[[ 0.45652774, -0.05273699,  0.09694741, ...,  0.08061957,
          0.0847721 , -0.03647762],
        [ 0.53588545, -0.6020879 , -0.52604985, ...,  0.29653946,
          0.15554446,  0.01623693],
        [ 0.4893116 , -0.18050799, -0.7247568 , ...,  0.03686034,
         -0.13996553,  0.04942406],
        ...,
        [-0.19481438, -0.4088347 ,  0.29001424, ..., -0.5040369 ,
          0.17912908,  0.02215274],
        [-0.16852173, -0.19396524,  0.03744941, ..., -0.0715602 ,
          0.14199317,  0.06604921],
        [-0.10381703,  0.00528723,  0.10848232, ...,  0.2147911 ,
          0.19264589, -0.05139716]]], dtype=float32)